# 07 — Multiple runs: одна job version, много запусков

Один и тот же SQL (без параметров, без литералов в фильтрах) прогоняется 5 раз. Логический план идентичен → jobVersion **не меняется**. Каждая итерация = новый `RunEvent` (START + COMPLETE) с одним и тем же jobVersion. В Marquez UI у job появится лента runs.

> Почему без `WHERE country = '...'`: OL-Spark считает версию плана с учётом литералов. Любая смена литерала в `WHERE` (например, `'RU' → 'US'`) приведёт к новой jobVersion. Чтобы реально получить **одну** version с N runs — план должен быть полностью идентичен между запусками.

Prerequisite: прогнан `00_setup.ipynb`. **Restart Jupyter kernel** перед запуском.

In [1]:
try:
    spark.stop()
except NameError:
    pass

from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .appName('lineage_07_multiple_runs')
    .enableHiveSupport()
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
assert spark.sparkContext.appName == 'lineage_07_multiple_runs', \
    f'appName fallback to {spark.sparkContext.appName!r} — restart Jupyter kernel'
print('app name:', spark.sparkContext.appName)
spark.sql('DROP TABLE IF EXISTS default.daily_country_totals')

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/27 13:12:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/27 13:12:24 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.
26/05/27 13:12:33 WARN BlackbirdModule: Unable to find Java 9+ MethodHandles.privateLookupIn.  Blackbird is not performing optimally!
26/05/27 13:12:33 WARN SparkApplicationNameApplicationJobNameProvider: Failed to obtain the application name. Using the default value [unknown]. Set the [spark.app.name] or [spark.openlineage.appName] property.
26/05/27 13:12:33 WARN SparkApplicationNameApplicationJobNameProvider: Failed to obtain the application name. Using the default value [unknown]. Set the [spark.app.name] or [spark.openlineage.appName] property.


app name: lineage_07_multiple_runs


26/05/27 13:12:40 WARN HiveConf: HiveConf of name hive.metastore.event.db.notification.api.auth does not exist
26/05/27 13:12:40 WARN HiveConf: HiveConf of name hive.log.dir does not exist
26/05/27 13:12:40 WARN HiveConf: HiveConf of name hive.root.logger does not exist
26/05/27 13:12:40 WARN HiveClientImpl: Detected HiveConf hive.execution.engine is 'tez' and will be reset to 'mr' to disable useless hive logic


DataFrame[]

In [2]:
import time

SQL = '''
    SELECT
        c.country,
        count(*)         AS orders,
        sum(o.amount)    AS revenue
    FROM default.raw_orders o
    JOIN default.raw_customers c ON c.id = o.customer_id
    GROUP BY c.country
'''

for i in range(1, 6):
    df = spark.sql(SQL)
    df.write.mode('overwrite').saveAsTable('default.daily_country_totals')
    print(f'run {i}: rows={spark.table("default.daily_country_totals").count()}')
    time.sleep(2)  # чтобы run-timestamps были визуально разнесены

26/05/27 13:12:41 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.analysis.ResolvedIdentifier
26/05/27 13:12:41 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.analysis.ResolvedIdentifier
26/05/27 13:12:41 WARN OutputStatisticsOutputDatasetFacetBuilder: No jobId found in context
26/05/27 13:12:41 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.analysis.ResolvedIdentifier
26/05/27 13:12:41 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.analysis.ResolvedIdentifier
26/05/27 13:12:47 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


run 1: rows=4
run 2: rows=4


26/05/27 13:12:50 ERROR ContextFactory: Query execution is null: can't emit event for executionId 6
26/05/27 13:12:50 ERROR ContextFactory: Query execution is null: can't emit event for executionId 6
26/05/27 13:12:50 ERROR ContextFactory: Query execution is null: can't emit event for executionId 6


run 3: rows=4


26/05/27 13:12:53 ERROR ContextFactory: Query execution is null: can't emit event for executionId 9
26/05/27 13:12:53 ERROR ContextFactory: Query execution is null: can't emit event for executionId 9
26/05/27 13:12:53 ERROR ContextFactory: Query execution is null: can't emit event for executionId 9


run 4: rows=4


26/05/27 13:12:56 ERROR ContextFactory: Query execution is null: can't emit event for executionId 12
26/05/27 13:12:56 ERROR ContextFactory: Query execution is null: can't emit event for executionId 12
26/05/27 13:12:56 ERROR ContextFactory: Query execution is null: can't emit event for executionId 12


run 5: rows=4


26/05/27 13:12:59 ERROR ContextFactory: Query execution is null: can't emit event for executionId 15
26/05/27 13:12:59 ERROR ContextFactory: Query execution is null: can't emit event for executionId 15
26/05/27 13:12:59 ERROR ContextFactory: Query execution is null: can't emit event for executionId 15


In [4]:
spark.stop()

## Что смотреть в Marquez

1. UI → jobs → найти job, связанный с `default.daily_country_totals` (app `lineage_07_multiple_runs`).
2. В detail tab **Runs** должна быть лента из 5 successful runs, все указывают на **один и тот же jobVersion**.
3. API:
   ```bash
   JOB=$(curl -s 'http://localhost:5000/api/v1/namespaces/hadoop-cluster/jobs' \
     | jq -r '.jobs[] | select(.name | contains("daily_country_totals")) | .name')
   curl -s "http://localhost:5000/api/v1/namespaces/hadoop-cluster/jobs/${JOB}/runs" | jq '.runs | length'
   ```
   Должно вернуть `5`.
4. Проверка единства jobVersion:
   ```bash
   curl -s "http://localhost:5000/api/v1/namespaces/hadoop-cluster/jobs/${JOB}/runs" \
     | jq '[.runs[].jobVersion.version] | unique | length'
   ```
   Должно вернуть `1`.